# 18.3 监控与可观测性 (Monitoring & Observability)

> 🕐 预估学习时间：35分钟

模型上线后，持续的监控与可观测性是保障服务质量的关键。本节涵盖推理性能监控、数据漂移检测、模型质量漂移检测以及告警与SLO管理。

本节涵盖：
- 监控维度总览（推理延迟、吞吐量、错误率、资源利用率、数据漂移、模型质量漂移）
- 推理性能监控（P50/P95/P99延迟、QPS、错误率统计与告警）
- 数据漂移检测（PSI、KL散度、JS散度）
- 模型质量漂移检测（预测分布变化、准确率下降）
- 告警系统与SLO管理（阈值告警、SLO定义、burn rate）

## 1. 监控维度总览

生产环境中的LLM服务需要从多个维度进行监控，形成完整的可观测性体系。

| 维度 | 关键指标 | 监控目的 |
|------|----------|----------|
| 推理性能 | P50/P95/P99延迟、QPS、错误率 | 保障用户体验 |
| 资源利用 | GPU利用率、显存占用、CPU、网络IO | 成本优化与容量规划 |
| 数据质量 | 输入分布漂移、特征统计、异常值 | 检测输入变化 |
| 模型质量 | 预测分布漂移、准确率、置信度 | 检测模型退化 |

**可观测性三大支柱**：
- **Metrics（指标）**：数值型时序数据，如延迟、QPS
- **Logs（日志）**：离散事件记录，如错误日志、请求日志
- **Traces（追踪）**：请求链路追踪，定位瓶颈

In [ ]:
import torch
import numpy as np
import random

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

monitoring_dimensions = {
    '推理性能': ['P50/P95/P99延迟', 'QPS吞吐量', '错误率', '超时率'],
    '资源利用': ['GPU利用率', '显存占用', 'CPU使用率', '网络IO'],
    '数据质量': ['输入分布漂移', '特征统计量', '数据新鲜度', '异常值比例'],
    '模型质量': ['预测分布漂移', '准确率下降', '置信度偏移', '输出长度变化'],
}

print('=== 监控维度总览 ===')
for category, metrics in monitoring_dimensions.items():
    print(f'\n【{category}】')
    for m in metrics:
        print(f'  - {m}')

print('\n--- 模拟指标快照 ---')
snapshot = {
    'p95_latency_ms': random.uniform(80, 120),
    'qps': random.uniform(800, 1200),
    'error_rate': random.uniform(0.001, 0.01),
    'gpu_util': random.uniform(0.6, 0.9),
    'psi_score': random.uniform(0.05, 0.25),
    'accuracy': random.uniform(0.88, 0.95),
}
for k, v in snapshot.items():
    print(f'  {k}: {v:.4f}')

print(f'\nKey: 监控需覆盖性能、资源、数据、模型四大维度，形成闭环可观测性。')

## 2. 推理性能监控

推理性能是用户最直接感知的服务质量指标。

**核心指标**：
- **延迟分位数（P50/P95/P99）**：比平均值更能反映尾部用户体验
  - P50：中位数，50%的请求低于此值
  - P95：95%的请求低于此值，反映大多数用户体验
  - P99：99%的请求低于此值，反映尾部延迟
- **QPS（Queries Per Second）**：系统吞吐量
- **错误率**：失败请求占比

**告警阈值示例**：
- P95延迟 > 200ms → 警告
- P99延迟 > 500ms → 严重
- 错误率 > 1% → 警告
- 错误率 > 5% → 严重

In [ ]:
import torch
import numpy as np
import time
from collections import deque

torch.manual_seed(42)
np.random.seed(42)

class MetricsCollector:
    def __init__(self, window_size=1000):
        self.window_size = window_size
        self.latencies = deque(maxlen=window_size)
        self.errors = deque(maxlen=window_size)
        self.timestamps = deque(maxlen=window_size)
        self.total_requests = 0
        self.total_errors = 0

    def record(self, latency_ms, is_error=False):
        self.latencies.append(latency_ms)
        self.errors.append(1 if is_error else 0)
        self.timestamps.append(time.time())
        self.total_requests += 1
        if is_error:
            self.total_errors += 1

    def percentile(self, data, p):
        if not data:
            return 0.0
        sorted_data = sorted(data)
        idx = int(len(sorted_data) * p / 100)
        idx = min(idx, len(sorted_data) - 1)
        return sorted_data[idx]

    def compute_qps(self):
        if len(self.timestamps) < 2:
            return 0.0
        duration = self.timestamps[-1] - self.timestamps[0]
        if duration <= 0:
            return 0.0
        return len(self.timestamps) / duration

    def summary(self):
        latencies = list(self.latencies)
        errors = list(self.errors)
        return {
            'count': len(latencies),
            'p50_ms': self.percentile(latencies, 50),
            'p95_ms': self.percentile(latencies, 95),
            'p99_ms': self.percentile(latencies, 99),
            'avg_ms': sum(latencies) / len(latencies) if latencies else 0,
            'qps': self.compute_qps(),
            'error_rate': sum(errors) / len(errors) if errors else 0,
        }

    def check_alerts(self, thresholds=None):
        thresholds = thresholds or {
            'p95_ms': 200, 'p99_ms': 500, 'error_rate': 0.05
        }
        s = self.summary()
        alerts = []
        for key, threshold in thresholds.items():
            value = s[key]
            if value > threshold:
                alerts.append((key, value, threshold))
        return alerts

collector = MetricsCollector(window_size=500)

for _ in range(300):
    latency = np.random.exponential(80) + np.random.uniform(0, 40)
    is_error = np.random.random() < 0.02
    collector.record(latency, is_error)
    time.sleep(0.001)

print('=== 推理性能监控 ===')
s = collector.summary()
count = s['count']
p50 = s['p50_ms']
p95 = s['p95_ms']
p99 = s['p99_ms']
avg = s['avg_ms']
qps = s['qps']
err = s['error_rate']
print(f'\n请求数: {count}')
print(f'延迟 P50: {p50:.2f}ms')
print(f'延迟 P95: {p95:.2f}ms')
print(f'延迟 P99: {p99:.2f}ms')
print(f'平均延迟: {avg:.2f}ms')
print(f'QPS: {qps:.1f}')
print(f'错误率: {err*100:.2f}%')

alerts = collector.check_alerts()
print(f'\n告警 ({len(alerts)} 条):')
for key, value, threshold in alerts:
    print(f'  [ALERT] {key}={value:.2f} > {threshold}')
if not alerts:
    print('  无告警，所有指标正常')

print(f'\nKey: P95/P99延迟比平均值更能反映尾部用户体验，是SLO的核心指标。')

## 3. 数据漂移检测

数据漂移指输入数据的分布随时间发生变化，可能导致模型性能下降。

**常用检测方法**：

| 方法 | 公式 | 特点 |
|------|------|------|
| PSI | Σ(curr% - ref%) × ln(curr%/ref%) | 行业标准，有明确阈值 |
| KL散度 | ΣP(x) × ln(P(x)/Q(x)) | 非对称，需相同支撑集 |
| JS散度 | 0.5×KL(P‖M) + 0.5×KL(Q‖M) | 对称有界，值域[0, ln2] |

**PSI判断标准**：
- PSI < 0.1：分布稳定
- 0.1 ≤ PSI < 0.25：轻微漂移，关注
- PSI ≥ 0.25：显著漂移，需重新训练

In [ ]:
import torch
import numpy as np

torch.manual_seed(42)
np.random.seed(42)

class DataDriftDetector:
    def __init__(self, reference_data, n_bins=10):
        self.reference = np.asarray(reference_data)
        self.n_bins = n_bins
        self.ref_hist, self.bin_edges = np.histogram(self.reference, bins=n_bins, density=True)
        self.ref_hist = self.ref_hist + 1e-10
        self.ref_hist = self.ref_hist / self.ref_hist.sum()

    def _to_hist(self, data):
        hist, _ = np.histogram(data, bins=self.bin_edges, density=True)
        hist = hist + 1e-10
        return hist / hist.sum()

    def psi(self, current_data):
        curr = self._to_hist(current_data)
        return np.sum((curr - self.ref_hist) * np.log(curr / self.ref_hist))

    def kl_divergence(self, current_data):
        curr = self._to_hist(current_data)
        return np.sum(self.ref_hist * np.log(self.ref_hist / curr))

    def js_divergence(self, current_data):
        curr = self._to_hist(current_data)
        m = 0.5 * (self.ref_hist + curr)
        kl_p = np.sum(self.ref_hist * np.log(self.ref_hist / m))
        kl_q = np.sum(curr * np.log(curr / m))
        return 0.5 * kl_p + 0.5 * kl_q

    def detect(self, current_data):
        return {
            'psi': self.psi(current_data),
            'kl': self.kl_divergence(current_data),
            'js': self.js_divergence(current_data),
        }

reference = np.random.normal(loc=0.0, scale=1.0, size=5000)

np.random.seed(42)
no_drift = np.random.normal(loc=0.0, scale=1.0, size=1000)
small_drift = np.random.normal(loc=0.3, scale=1.1, size=1000)
large_drift = np.random.normal(loc=1.0, scale=1.5, size=1000)

detector = DataDriftDetector(reference, n_bins=20)

print('=== 数据漂移检测 ===')
print(f'\n参考分布: mean={reference.mean():.3f}, std={reference.std():.3f}')

scenarios = [('无漂移', no_drift), ('轻微漂移', small_drift), ('严重漂移', large_drift)]
for name, data in scenarios:
    scores = detector.detect(data)
    psi = scores['psi']
    kl = scores['kl']
    js = scores['js']
    if psi < 0.1:
        psi_status = '< 0.1 稳定'
    elif psi >= 0.25:
        psi_status = '>= 0.25 严重漂移'
    else:
        psi_status = '0.1-0.25 轻微漂移'
    print(f'\n【{name}】 mean={data.mean():.3f}, std={data.std():.3f}')
    print(f'  PSI: {psi:.4f}  {psi_status}')
    print(f'  KL : {kl:.4f}')
    print(f'  JS : {js:.4f}')

print(f'\nKey: PSI<0.1稳定，0.1-0.25轻微漂移，>0.25严重漂移；JS散度对称且有界。')

## 4. 模型质量漂移检测

即使输入分布不变，模型预测质量也可能因环境变化而退化。

**检测维度**：
- **预测分布漂移**：模型输出分布是否发生变化
- **准确率下降**：在线准确率是否低于基线
- **置信度偏移**：模型预测置信度的统计量变化
- **输出长度变化**：生成长度的分布偏移

**早期预警价值**：
预测分布漂移通常先于准确率明显下降出现，可作为模型退化的早期信号，争取重新训练的时间窗口。

In [ ]:
import torch
import numpy as np
from collections import deque

torch.manual_seed(42)
np.random.seed(42)

class PredictionDriftDetector:
    def __init__(self, reference_predictions, n_bins=10):
        self.reference = np.asarray(reference_predictions)
        self.n_bins = n_bins
        self.ref_hist, self.bin_edges = np.histogram(self.reference, bins=n_bins, density=True)
        self.ref_hist = self.ref_hist + 1e-10
        self.ref_hist = self.ref_hist / self.ref_hist.sum()
        self.ref_mean = self.reference.mean()
        self.ref_std = self.reference.std()
        self.accuracy_history = deque(maxlen=100)
        self.ref_accuracy = None

    def set_baseline_accuracy(self, acc):
        self.ref_accuracy = acc

    def prediction_psi(self, current_predictions):
        hist, _ = np.histogram(current_predictions, bins=self.bin_edges, density=True)
        hist = hist + 1e-10
        hist = hist / hist.sum()
        return np.sum((hist - self.ref_hist) * np.log(hist / self.ref_hist))

    def record_accuracy(self, acc):
        self.accuracy_history.append(acc)

    def accuracy_drop(self):
        if not self.accuracy_history or self.ref_accuracy is None:
            return 0.0
        recent = list(self.accuracy_history)[-10:]
        avg_recent = sum(recent) / len(recent)
        return self.ref_accuracy - avg_recent

    def detect(self, current_predictions, current_accuracy=None):
        psi = self.prediction_psi(current_predictions)
        result = {
            'pred_psi': psi,
            'mean_shift': current_predictions.mean() - self.ref_mean,
            'std_shift': current_predictions.std() - self.ref_std,
        }
        if current_accuracy is not None:
            self.record_accuracy(current_accuracy)
            result['accuracy_drop'] = self.accuracy_drop()
        return result

np.random.seed(42)
ref_preds = np.random.beta(5, 2, size=5000)
ref_accuracy = 0.92

detector = PredictionDriftDetector(ref_preds, n_bins=20)
detector.set_baseline_accuracy(ref_accuracy)

print('=== 模型质量漂移检测 ===')
print(f'\n参考预测: mean={ref_preds.mean():.3f}, std={ref_preds.std():.3f}')
print(f'基线准确率: {ref_accuracy:.3f}')

np.random.seed(123)
for week in range(4):
    a_param = max(0.5, 5 - week * 0.8)
    b_param = 2 + week * 0.5
    curr_preds = np.random.beta(a_param, b_param, size=500)
    curr_acc = ref_accuracy - week * 0.015 - np.random.uniform(0, 0.01)
    result = detector.detect(curr_preds, curr_acc)
    psi = result['pred_psi']
    mean_shift = result['mean_shift']
    acc_drop = result['accuracy_drop']
    print(f'\n第 {week+1} 周:')
    print(f'  预测PSI: {psi:.4f}')
    print(f'  均值偏移: {mean_shift:+.4f}')
    print(f'  准确率下降: {acc_drop:.4f}')
    if psi > 0.2:
        print(f'  [警告] 预测分布显著漂移')

print(f'\nKey: 预测分布漂移常先于准确率下降出现，是早期预警信号。')

## 5. 告警系统与SLO管理

**SLO（Service Level Objective）**：服务可用性目标，如99.9%的请求P95延迟<200ms。

**错误预算（Error Budget）**：
- 可用性99.9% → 错误预算0.1%
- 错误预算消耗完毕前可以放心发布新版本

**Burn Rate（燃烧速率）**：
- burn rate = 当前错误率 / 错误预算
- burn rate > 1：消耗错误预算过快，需介入
- burn rate > 2：严重，立即处理

**告警分级**：
- **Warning**：指标超过警告阈值，关注但不需立即处理
- **Critical**：指标超过严重阈值，需立即介入
- 多窗口告警（如1小时+6小时）减少误报

In [ ]:
import torch
import numpy as np
from dataclasses import dataclass

torch.manual_seed(42)
np.random.seed(42)

@dataclass
class SLO:
    name: str
    target: float
    window_days: int
    error_budget: float

@dataclass
class Alert:
    severity: str
    metric: str
    value: float
    threshold: float
    message: str

class AlertManager:
    def __init__(self):
        self.slos = {}
        self.alert_rules = []
        self.fired_alerts = []

    def add_slo(self, slo):
        self.slos[slo.name] = slo

    def add_rule(self, metric, threshold, severity='warning', message=''):
        self.alert_rules.append({
            'metric': metric, 'threshold': threshold,
            'severity': severity, 'message': message,
        })

    def evaluate(self, metrics):
        new_alerts = []
        for rule in self.alert_rules:
            metric = rule['metric']
            if metric not in metrics:
                continue
            value = metrics[metric]
            if value > rule['threshold']:
                alert = Alert(
                    severity=rule['severity'],
                    metric=metric,
                    value=value,
                    threshold=rule['threshold'],
                    message=rule['message'] or f'{metric} 超过阈值',
                )
                new_alerts.append(alert)
                self.fired_alerts.append(alert)
        return new_alerts

    def compute_burn_rate(self, slo_name, current_error_rate):
        slo = self.slos.get(slo_name)
        if not slo or slo.error_budget <= 0:
            return 0.0
        return current_error_rate / slo.error_budget

    def slo_status(self, slo_name, current_error_rate):
        slo = self.slos.get(slo_name)
        if not slo:
            return '未知'
        burn = self.compute_burn_rate(slo_name, current_error_rate)
        if burn > 1:
            return f'危险 (burn rate={burn:.2f}, 超出错误预算)'
        elif burn > 0.5:
            return f'警告 (burn rate={burn:.2f})'
        else:
            return f'正常 (burn rate={burn:.2f})'

manager = AlertManager()

manager.add_slo(SLO('推理可用性', target=0.999, window_days=30, error_budget=0.001))
manager.add_slo(SLO('推理延迟', target=0.95, window_days=30, error_budget=0.05))

manager.add_rule('p95_latency_ms', 200, 'warning', 'P95延迟超过200ms')
manager.add_rule('p95_latency_ms', 500, 'critical', 'P95延迟超过500ms')
manager.add_rule('error_rate', 0.01, 'warning', '错误率超过1%')
manager.add_rule('error_rate', 0.05, 'critical', '错误率超过5%')
manager.add_rule('psi', 0.25, 'critical', '数据漂移严重')

print('=== 告警系统与SLO管理 ===')
print('\n--- SLO定义 ---')
for name, slo in manager.slos.items():
    print(f'  {name}: 目标={slo.target}, 错误预算={slo.error_budget}')

np.random.seed(42)
print('\n--- 监控时间线 ---')
for t in range(5):
    metrics = {
        'p95_latency_ms': np.random.uniform(100, 600),
        'error_rate': np.random.uniform(0.001, 0.06),
        'psi': np.random.uniform(0.05, 0.3),
    }
    alerts = manager.evaluate(metrics)
    p95 = metrics['p95_latency_ms']
    err = metrics['error_rate']
    psi = metrics['psi']
    print(f'\nT{t}: p95={p95:.1f}ms, err={err*100:.2f}%, psi={psi:.3f}')
    status = manager.slo_status('推理可用性', err)
    print(f'  SLO状态: {status}')
    if alerts:
        for a in alerts:
            sev = a.severity.upper()
            print(f'  [{sev}] {a.message} ({a.value:.3f} > {a.threshold})')
    else:
        print(f'  无告警')

print(f'\nKey: Burn rate>1表示消耗错误预算过快，需立即介入；多窗口告警减少误报。')

## 📝 课后思考题

1. 在LLM推理服务中，P99延迟突然升高但P50正常，可能的原因有哪些？如何排查？
2. 如果检测到数据漂移但模型准确率未下降，是否需要重新训练？请分析利弊。
3. 如何设计多窗口告警策略（如1小时+6小时+24小时）来平衡告警的敏感性与误报率？
4. 错误预算（Error Budget）耗尽后，应该停止发布新版本还是降低SLO目标？请说明你的理由。